# Run the pilot on a Colab GPU

First: Runtime -> Change runtime type -> T4 GPU.

This clones the repo, downloads CIFAR-10-C, and runs the phase-1 check (teacher
+ w0.5 student, fp32, seeds 0 and 1, 30 epochs): train, evaluate on CIFAR-10 and
CIFAR-10-C, measure geometry, aggregate. About 15-25 minutes on a T4. The full
18-run pilot is described in the last cell.

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
%cd /content
![ -d student-teacher-landscape ] || git clone -b dev https://github.com/Iyeba-Kallon/student-teacher-landscape.git
%cd /content/student-teacher-landscape
!git pull --quiet

In [ ]:
# Colab already has torch, numpy, pandas, matplotlib. pyhessian is the only gap.
!pip install -q pyhessian
import pyhessian, yaml, torchvision
print('deps ok')

In [ ]:
# CIFAR-10-C, about 2.9 GB. Idempotent. CIFAR-10 downloads itself during training.
!python data/download_cifar10c.py --dest data/

In [ ]:
!bash scripts/phase1.sh

In [ ]:
import pandas as pd
df = pd.read_csv('results/pilot_summary.csv')
df[['run_name', 'mode', 'width_mult', 'seed', 'id_acc', 'ood_acc_mean',
    'mce_vs_baseline', 'adaptive_sharpness', 'hessian_trace',
    'hessian_top_eigenvalue']]

In [ ]:
# Per seed: does the student beat its teacher on OOD, and is it flatter?
t = df[df['mode'] == 'teacher'].set_index('seed')
for _, s in df[df['mode'] == 'student'].iterrows():
    ts = t.loc[s['seed']]
    ood = 'beats' if s['ood_acc_mean'] > ts['ood_acc_mean'] else 'below'
    flat = 'flatter' if s['adaptive_sharpness'] < ts['adaptive_sharpness'] else 'sharper'
    print(f"seed {s['seed']}  {s['run_name']:24s} "
          f"OOD {s['ood_acc_mean']:.3f} vs {ts['ood_acc_mean']:.3f} ({ood})  "
          f"sharpness {s['adaptive_sharpness']:.3f} vs {ts['adaptive_sharpness']:.3f} ({flat})")

In [ ]:
# Zip the results and download them.
!zip -qr results_phase1.zip results/ -x 'results/**/checkpoints/*'
from google.colab import files
files.download('results_phase1.zip')

## Full pilot

If phase-1 shows the pattern, set `epochs` back to 200 in
`configs/teacher_fp32.yaml` and `configs/student_w0.5_fp32.yaml` and run
`!SKIP_DATA=1 bash scripts/run_pilot.sh`.

That is 18 training runs plus evaluate and geometry, roughly 12-18 hours on one
T4, longer than a free Colab session. Ways around it:

- Kaggle gives 12-hour sessions and 30 GPU-hours a week; `run_pilot.sh` and
  `phase1.sh` skip finished training runs, so a second session continues.
- Keep `results/` on Drive so a killed session resumes:
  `from google.colab import drive; drive.mount('/content/drive')` then
  `!mkdir -p /content/drive/MyDrive/stl_results && ln -sfn /content/drive/MyDrive/stl_results results`.
- Run `SKIP_AMP=1` first for the fp32 half, then the AMP configs.

Bring back `results/pilot_summary.csv` and the per-run json files and open
`notebooks/analysis.ipynb` locally.